# Extra — A market Q&A chat with RAG (Chroma + Gemini)

**Why this earns its place:** a dashboard tab where a user asks *"what does this
stock do?"* or *"what is a moving average?"* and gets an answer grounded in your
own notes — not the model's guesses. That grounding is **RAG**: retrieve relevant
notes, then let the model answer using them.

Runs in **MOCK mode** with no key (retrieval only). Add a Gemini key for full answers.

## Setup

In [ ]:
import os
API_KEY = os.environ.get('GEMINI_API_KEY')
USE_MOCK = API_KEY is None
print('MOCK mode (retrieval only)' if USE_MOCK else 'using Gemini for answers')

## A tiny market knowledge base
Replace with company filings, an EGX primer, your strategy notes.

In [ ]:
docs = [
  'A moving average smooths price by averaging the last N days; crossovers signal trend changes.',
  'The EGX30 is the main Egyptian stock index, tracking 30 large listed companies.',
  'CIB (COMI) is Egypt\'s largest private-sector bank.',
  'Volatility is the standard deviation of returns; higher volatility means higher risk.',
  'A benchmark is what you compare a strategy against; beating it is the goal.',
]

## Build a local vector store (Chroma)
Chroma embeds each note and lets us retrieve the most relevant ones for a question.
No cloud, no key.

In [ ]:
import numpy as np, chromadb
from chromadb import Documents, Embeddings, EmbeddingFunction

# A tiny no-download embedding so this runs offline anywhere. It's crude but
# enough to show the RAG flow. With internet you can drop the embedding_function
# argument to use Chroma's default (better) model, or use Gemini embeddings.
class HashEmbed(EmbeddingFunction):
    def __init__(self): pass
    def __call__(self, input: Documents) -> Embeddings:
        out = []
        for t in input:
            v = np.zeros(256)
            for tok in t.lower().split():
                v[hash(tok) % 256] += 1.0
            n = np.linalg.norm(v)
            out.append((v / n if n else v).tolist())
        return out

client = chromadb.Client()
try: client.delete_collection('market')
except Exception: pass
col = client.create_collection('market', embedding_function=HashEmbed())
col.add(documents=docs, ids=[f'd{i}' for i in range(len(docs))])
print('indexed', col.count(), 'notes')

## Retrieve

In [ ]:
def retrieve(question, k=2):
    res = col.query(query_texts=[question], n_results=k)
    return res['documents'][0]

q = 'what is a moving average?'
hits = retrieve(q)
print('retrieved:'); [print(' -', h) for h in hits]

## Answer (grounded)
Mock mode returns the retrieved notes. With a key, Gemini writes an answer using
only those notes.

In [ ]:
def answer(question):
    context = '\n'.join(retrieve(question))
    if USE_MOCK:
        return 'Based on the notes:\n' + context
    import google.generativeai as genai
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel('gemini-1.5-flash')
    prompt = ('Answer the question using ONLY these notes. If they do not cover it, '
              'say so.\nNotes:\n' + context + '\nQuestion: ' + question)
    return model.generate_content(prompt).text

print(answer('what is a moving average?'))
print('---')
print(answer('what is the EGX30?'))

## Wire into the dashboard
Add a `/ask` endpoint that calls `answer(question)`, and a chat box tab in the
frontend. Users get grounded market answers inside your product.